# Startup Founder Burnout 2026: End-to-End Data Pipeline

**Steps covered**
1. Setup & configuration
2. Load data and initial audit
3. Data cleaning (missing values, duplicates, formats, text, outliers) using Python **and** SQL (SQLite)
4. Validation checks (ranges and logical consistency)
5. Feature engineering: `Shutdown_Probability`, `Shutdown_Risk`, `Startup_Failure_Flag`
6. Exploratory data analysis (EDA)
7. Predictive modelling of `Startup_Failure_Flag` (with data-leakage protection)
8. Export of final files

> The same code works on your **raw** file and on the **already-cleaned** file. Every step is idempotent, so on clean data it simply changes nothing.

## 1. Setup & configuration

In [ ]:
import warnings
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, average_precision_score)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

# ---- CONFIG: change the path to your file (raw or cleaned) ----
DATA_PATH = Path("startup_founder_burnout_2026.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Business rules used for validation / feature creation
RISK_BINS = [-np.inf, 0.35, 0.65, np.inf]          # Low < 0.35 <= Medium <= 0.65 < High
RISK_LABELS = ["Low", "Medium", "High"]
BURNOUT_BINS = [-np.inf, 3.5, 6.5, np.inf]         # Low < 3.5 <= Moderate <= 6.5 < Severe
BURNOUT_LABELS = ["Low", "Moderate", "Severe"]

## 2. Load data and initial audit

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
def audit(df: pd.DataFrame) -> None:
    """Quick data-quality report."""
    print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
    print(f"Fully duplicated rows : {df.duplicated().sum():,}")
    miss = df.isna().sum()
    print(f"Columns with missing  : {(miss > 0).sum()} (total missing cells: {miss.sum():,})")
    if miss.sum():
        display(miss[miss > 0].sort_values(ascending=False).to_frame("missing")
                .assign(pct=lambda d: d["missing"] / len(df) * 100))

audit(raw)

In [ ]:
raw.describe(include="all").T

## 3. Data cleaning
### 3.1 Column groups

In [ ]:
CAT_COLS = ["Founder_Type", "Economic_Climate", "Industry", "Funding_Stage", "Work_Mode",
            "Burnout_Level", "Seeks_Mental_Health_Support", "Shutdown_Risk"]
INT_COLS = ["Founder_Age", "Founder_Experience_Years", "Team_Size", "Startup_Age_Months",
            "Founder_Burnout_Flag", "Startup_Failure_Flag"]
FLOAT_COLS = ["Weekly_Work_Hours", "Sleep_Hours", "Exercise_Days_Per_Week", "Vacation_Days_Taken",
              "Investor_Pressure_Score", "Cofounder_Conflict_Score", "Stress_Score",
              "Decision_Fatigue_Score", "Burnout_Score", "Monthly_Revenue_Growth_Percent",
              "Runway_Months_Remaining", "Product_Market_Fit_Score", "Employee_Turnover_Percent",
              "Work_Life_Balance_Score", "Shutdown_Probability"]

# Allowed ranges (min, max); values outside become NaN and are re-imputed
VALID_RANGES = {
    "Founder_Age": (18, 80), "Founder_Experience_Years": (0, 60), "Team_Size": (1, 100_000),
    "Startup_Age_Months": (0, 600), "Weekly_Work_Hours": (0, 168), "Sleep_Hours": (0, 24),
    "Exercise_Days_Per_Week": (0, 7), "Vacation_Days_Taken": (0, 365),
    "Investor_Pressure_Score": (0, 10), "Cofounder_Conflict_Score": (0, 10),
    "Stress_Score": (0, 10), "Decision_Fatigue_Score": (0, 10), "Burnout_Score": (0, 10),
    "Product_Market_Fit_Score": (0, 10), "Work_Life_Balance_Score": (0, 10),
    "Employee_Turnover_Percent": (0, 100), "Runway_Months_Remaining": (0, 120),
    "Monthly_Revenue_Growth_Percent": (-100, 1000), "Shutdown_Probability": (0, 1),
}

### 3.2 Python cleaning function
Handles: whitespace/case issues, wrong data types, impossible values, duplicates, missing values.

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Standardise column names
    df.columns = df.columns.str.strip().str.replace(r"\s+", "_", regex=True)

    # 2) Clean text columns: trim, collapse spaces, consistent Title Case, fake-null strings -> NaN
    null_tokens = {"", "nan", "none", "null", "n/a", "na", "-", "?", "unknown"}
    for c in [c for c in CAT_COLS if c in df.columns]:
        s = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
        s = s.mask(s.str.lower().isin(null_tokens))
        df[c] = s
    # Yes/No normalisation
    if "Seeks_Mental_Health_Support" in df:
        df["Seeks_Mental_Health_Support"] = (df["Seeks_Mental_Health_Support"].str.lower()
            .map({"yes": "Yes", "y": "Yes", "true": "Yes", "1": "Yes",
                  "no": "No", "n": "No", "false": "No", "0": "No"}))
    for c in ["Founder_Type", "Economic_Climate", "Industry", "Funding_Stage", "Work_Mode",
              "Burnout_Level", "Shutdown_Risk"]:
        if c in df:
            df[c] = df[c].str.title()
    # Keep well-known acronyms upper-case
    if "Industry" in df:
        df["Industry"] = df["Industry"].replace({"Ai": "AI", "Saas": "SaaS", "Fintech": "FinTech",
            "Healthtech": "HealthTech", "Edtech": "EdTech", "Climatetech": "ClimateTech",
            "E-Commerce": "E-commerce"})

    # 3) Fix numeric formats (strip %, commas, currency; coerce bad text to NaN)
    for c in [c for c in INT_COLS + FLOAT_COLS if c in df.columns]:
        if df[c].dtype == "object" or str(df[c].dtype).startswith("string"):
            df[c] = df[c].astype("string").str.replace(r"[,%$\s]", "", regex=True)
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 4) Impossible values -> NaN
    for c, (lo, hi) in VALID_RANGES.items():
        if c in df:
            df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan

    # 5) Remove duplicates (exact, then keep first)
    df = df.drop_duplicates().reset_index(drop=True)

    # 6) Missing values: median for numeric, mode for categorical
    #    (derived columns are NOT imputed. They are recomputed in step 5.)
    derived = {"Shutdown_Probability", "Shutdown_Risk", "Startup_Failure_Flag",
               "Burnout_Level", "Founder_Burnout_Flag"}
    for c in [c for c in FLOAT_COLS + INT_COLS if c in df and c not in derived]:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())
    for c in [c for c in CAT_COLS if c in df and c not in derived]:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].mode().iloc[0])

    # 7) Final dtypes
    for c in [c for c in INT_COLS if c in df and c not in derived]:
        df[c] = df[c].round().astype("int64")
    for c in [c for c in CAT_COLS if c in df]:
        df[c] = df[c].astype("object")
    return df

df = clean_data(raw)
print("Before:", raw.shape, "| After:", df.shape)
audit(df)

### 3.3 SQL cleaning (SQLite)
Same idea as your VS Code Python+SQL workflow: load into SQLite, de-duplicate and filter with SQL, then read back.

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("founders", conn, index=False, if_exists="replace")

sql_dedup = """
SELECT *
FROM founders
WHERE rowid IN (SELECT MIN(rowid) FROM founders GROUP BY {cols})   -- keep first of each duplicate group
  AND Weekly_Work_Hours BETWEEN 0 AND 168
  AND Sleep_Hours BETWEEN 0 AND 24
  AND Founder_Age BETWEEN 18 AND 80
""".format(cols=", ".join(f'"{c}"' for c in df.columns))

df_sql = pd.read_sql_query(sql_dedup, conn)
print("Rows after SQL de-dup + sanity filter:", len(df_sql))

# Example analytical SQL queries
pd.read_sql_query("""
    SELECT Funding_Stage,
           COUNT(*)                          AS founders,
           ROUND(AVG(Burnout_Score), 2)      AS avg_burnout,
           ROUND(AVG(Runway_Months_Remaining), 1) AS avg_runway
    FROM founders
    GROUP BY Funding_Stage
    ORDER BY avg_burnout DESC
""", conn)

In [ ]:
df = df_sql.copy()   # continue with the SQL-validated frame

### 3.4 Outlier review (flag, don't blindly delete)
`Team_Size` is heavily right-skewed. Real large startups exist, so we flag and log-transform instead of dropping.

In [ ]:
def iqr_outliers(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

num_cols = [c for c in FLOAT_COLS + INT_COLS
            if c in df and c not in {"Founder_Burnout_Flag", "Startup_Failure_Flag"}]
outlier_summary = pd.DataFrame({
    "outliers": {c: int(iqr_outliers(df[c]).sum()) for c in num_cols}
}).assign(pct=lambda d: d["outliers"] / len(df) * 100).sort_values("outliers", ascending=False)
outlier_summary.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df["Team_Size"], bins=50, ax=ax[0]); ax[0].set_title("Team_Size (raw)")
sns.histplot(np.log1p(df["Team_Size"]), bins=50, ax=ax[1]); ax[1].set_title("log1p(Team_Size)")
plt.tight_layout(); plt.show()

## 4. Validation checks

In [ ]:
checks = {}
checks["no missing values"] = df.isna().sum().sum() == 0
checks["no duplicate rows"] = df.duplicated().sum() == 0
checks["all values inside valid ranges"] = all(
    df[c].between(lo, hi).all() for c, (lo, hi) in VALID_RANGES.items() if c in df)
checks["experience <= age - 16"] = ((df["Founder_Age"] - df["Founder_Experience_Years"]) >= 16).mean() > 0.99
checks["binary flags are 0/1"] = df[["Founder_Burnout_Flag", "Startup_Failure_Flag"]].isin([0, 1]).all().all()

for k, v in checks.items():
    print(("PASS" if v else "FAIL"), "-", k)

# Rows where experience is larger than plausible given age (reported, not deleted)
suspect = df[(df["Founder_Age"] - df["Founder_Experience_Years"]) < 16]
print(f"\nRows with age - experience < 16: {len(suspect)} ({len(suspect)/len(df):.2%})")

## 5. Feature engineering
Derived columns:
* **Burnout_Level / Founder_Burnout_Flag** from `Burnout_Score` (Low < 3.5, Moderate 3.5–6.5, Severe > 6.5)
* **Shutdown_Probability**: your original formula is kept if the column exists. If it does not, a transparent logistic score is created (see note).
* **Shutdown_Risk**: Low < 0.35, Medium 0.35–0.65, High > 0.65
* **Startup_Failure_Flag**: 1 if `Shutdown_Risk == "High"`

> **Note:** I could not see your original formula for `Shutdown_Probability`, so the fallback below is an *illustrative* weighted logistic score built from the strongest drivers. Replace it with your own logic if you want an exact reproduction.

In [ ]:
def add_burnout_features(d: pd.DataFrame) -> pd.DataFrame:
    """Create Burnout_Level / Founder_Burnout_Flag only where missing (your originals are kept)."""
    d = d.copy()
    level = pd.cut(d["Burnout_Score"], BURNOUT_BINS, labels=BURNOUT_LABELS, right=True).astype(str)
    if "Burnout_Level" not in d.columns:
        d["Burnout_Level"] = level
    else:
        d["Burnout_Level"] = d["Burnout_Level"].fillna(level)
    if "Founder_Burnout_Flag" not in d.columns:
        d["Founder_Burnout_Flag"] = (d["Burnout_Level"] == "Severe").astype(int)
    return d

def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

def compute_shutdown_probability(d: pd.DataFrame) -> pd.Series:
    """Illustrative logistic risk score (fallback when the column is missing)."""
    z = (0.60 * zscore(d["Burnout_Score"]) + 0.50 * zscore(d["Employee_Turnover_Percent"])
         + 0.45 * zscore(d["Cofounder_Conflict_Score"]) + 0.40 * zscore(d["Weekly_Work_Hours"])
         + 0.30 * zscore(d["Investor_Pressure_Score"])
         - 0.60 * zscore(d["Work_Life_Balance_Score"]) - 0.55 * zscore(d["Runway_Months_Remaining"])
         - 0.50 * zscore(d["Product_Market_Fit_Score"]) - 0.30 * zscore(d["Sleep_Hours"])
         - 0.20 * zscore(d["Monthly_Revenue_Growth_Percent"]))
    z = z - 0.9                       # shift so ~1/3 of startups are high-ish risk
    return 1 / (1 + np.exp(-z))

def add_shutdown_features(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    if "Shutdown_Probability" not in d.columns or d["Shutdown_Probability"].isna().any():
        d["Shutdown_Probability"] = compute_shutdown_probability(d)
    p = d["Shutdown_Probability"]
    d["Shutdown_Risk"] = pd.cut(p, RISK_BINS, labels=RISK_LABELS, right=True).astype(str)
    d["Startup_Failure_Flag"] = (d["Shutdown_Risk"] == "High").astype(int)
    d["Shutdown_Probability"] = p.round(3)
    return d

# To keep YOUR original values, we only fill what is missing; Risk & Flag are verified below.
orig_risk, orig_flag = df["Shutdown_Risk"].copy(), df["Startup_Failure_Flag"].copy()
df_feat = add_burnout_features(df)
df_feat = add_shutdown_features(df_feat)

# Agreement with your existing labels (small differences happen only on boundary values
# because the original bins were applied before the probability was rounded to 3 decimals)
print("Shutdown_Risk agreement      :", f"{(df_feat['Shutdown_Risk'] == orig_risk).mean():.3%}")
print("Startup_Failure_Flag agreement:", f"{(df_feat['Startup_Failure_Flag'] == orig_flag).mean():.3%}")

# Keep original labels as the source of truth when present
df_feat["Shutdown_Risk"] = orig_risk.values
df_feat["Startup_Failure_Flag"] = orig_flag.values
df = df_feat

### Additional engineered features (used for EDA / modelling)
Ratio and interaction features add useful signal. **None of them use the target or its parents.**

In [ ]:
df["Log_Team_Size"] = np.log1p(df["Team_Size"])
df["Experience_Ratio"] = df["Founder_Experience_Years"] / (df["Founder_Age"] - 17).clip(lower=1)
df["Sleep_Deficit"] = (7 - df["Sleep_Hours"]).clip(lower=0)
df["Overwork_Flag"] = (df["Weekly_Work_Hours"] > 60).astype(int)
df["Stress_Load_Index"] = df[["Stress_Score", "Decision_Fatigue_Score",
                              "Investor_Pressure_Score", "Cofounder_Conflict_Score"]].mean(axis=1)
df["Recovery_Index"] = (df["Sleep_Hours"] / 8 + df["Exercise_Days_Per_Week"] / 7
                        + df["Vacation_Days_Taken"] / 30 + df["Work_Life_Balance_Score"] / 10) / 4
df["Burn_Rate_Pressure"] = 1 / df["Runway_Months_Remaining"].clip(lower=1)
df["Startup_Stage_Group"] = pd.cut(df["Startup_Age_Months"], [0, 24, 60, 120, np.inf],
                                   labels=["<2y", "2-5y", "5-10y", "10y+"]).astype(str)
df[["Log_Team_Size", "Experience_Ratio", "Sleep_Deficit", "Stress_Load_Index",
    "Recovery_Index", "Burn_Rate_Pressure"]].describe().T

## 6. Exploratory data analysis

### 6.1 Target and category balance

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
df["Startup_Failure_Flag"].value_counts().sort_index().plot.bar(ax=ax[0], color=["#4C9F70", "#D1495B"])
ax[0].set_title("Startup_Failure_Flag"); ax[0].set_xticklabels(["0 = Survives", "1 = Fails"], rotation=0)
df["Shutdown_Risk"].value_counts().reindex(RISK_LABELS).plot.bar(ax=ax[1], color=["#4C9F70", "#EDAE49", "#D1495B"])
ax[1].set_title("Shutdown_Risk"); ax[1].tick_params(axis="x", rotation=0)
df["Burnout_Level"].value_counts().reindex(BURNOUT_LABELS).plot.bar(ax=ax[2], color=["#4C9F70", "#EDAE49", "#D1495B"])
ax[2].set_title("Burnout_Level"); ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print(f"Failure rate: {df['Startup_Failure_Flag'].mean():.2%}")

### 6.2 Distributions of numeric variables

In [ ]:
dist_cols = ["Founder_Age", "Weekly_Work_Hours", "Sleep_Hours", "Burnout_Score", "Stress_Score",
             "Runway_Months_Remaining", "Product_Market_Fit_Score", "Employee_Turnover_Percent",
             "Work_Life_Balance_Score", "Shutdown_Probability", "Monthly_Revenue_Growth_Percent",
             "Vacation_Days_Taken"]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for a, c in zip(axes.ravel(), dist_cols):
    sns.histplot(df[c], bins=40, kde=True, ax=a, color="#3E7CB1"); a.set_title(c, fontsize=10)
    a.set_xlabel("")
plt.tight_layout(); plt.show()

### 6.3 Correlation heatmap
`Shutdown_Probability` is shown here for insight only. It is **not** a legitimate model input (see leakage note below).

In [ ]:
corr_cols = [c for c in df.select_dtypes("number").columns if c not in {"Overwork_Flag"}]
corr = df[corr_cols].corr()
plt.figure(figsize=(15, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".1f", annot_kws={"size": 6}, linewidths=.3)
plt.title("Correlation matrix"); plt.show()

corr["Shutdown_Probability"].drop("Shutdown_Probability").sort_values().to_frame("corr").T

### 6.4 Failure rate by segment

In [ ]:
cat_eda = ["Founder_Type", "Economic_Climate", "Industry", "Funding_Stage", "Work_Mode",
           "Seeks_Mental_Health_Support", "Burnout_Level", "Startup_Stage_Group"]
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for a, c in zip(axes.ravel(), cat_eda):
    rate = df.groupby(c)["Startup_Failure_Flag"].mean().sort_values(ascending=False)
    sns.barplot(x=rate.values, y=rate.index, ax=a, color="#D1495B")
    a.axvline(df["Startup_Failure_Flag"].mean(), color="k", ls="--", lw=1)
    a.set_title(f"Failure rate by {c}"); a.set_xlabel("failure rate"); a.set_ylabel("")
plt.tight_layout(); plt.show()

### 6.5 Burnout vs shutdown risk

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
sns.boxplot(data=df, x="Burnout_Level", y="Shutdown_Probability", order=BURNOUT_LABELS, ax=ax[0])
sns.scatterplot(data=df.sample(5000, random_state=RANDOM_STATE), x="Burnout_Score", y="Shutdown_Probability",
                hue="Shutdown_Risk", hue_order=RISK_LABELS, alpha=.5, s=12, ax=ax[1])
sns.heatmap(pd.crosstab(df["Burnout_Level"], df["Shutdown_Risk"], normalize="index")
            .reindex(index=BURNOUT_LABELS, columns=RISK_LABELS), annot=True, fmt=".0%", cmap="Reds", ax=ax[2])
ax[0].set_title("Shutdown probability by burnout level"); ax[1].set_title("Burnout vs shutdown probability")
ax[2].set_title("Risk mix within each burnout level")
plt.tight_layout(); plt.show()

### 6.6 Statistical tests (is burnout linked to failure?)

In [ ]:
from scipy import stats

grp0 = df.loc[df["Startup_Failure_Flag"] == 0, "Burnout_Score"]
grp1 = df.loc[df["Startup_Failure_Flag"] == 1, "Burnout_Score"]
t, p = stats.mannwhitneyu(grp0, grp1)
print(f"Mann-Whitney U (Burnout_Score, failed vs survived): p = {p:.2e}")

chi2, p, *_ = stats.chi2_contingency(pd.crosstab(df["Funding_Stage"], df["Startup_Failure_Flag"]))
print(f"Chi-square (Funding_Stage vs failure): chi2 = {chi2:,.1f}, p = {p:.2e}")

## 7. Predictive modelling: `Startup_Failure_Flag`

### Data-leakage warning
`Shutdown_Risk` and `Startup_Failure_Flag` are **direct deterministic functions** of `Shutdown_Probability`
(Flag = 1 when probability > 0.65). Using any of the three as an input would give a fake ~100% score.
They are therefore excluded. `Burnout_Level` and `Founder_Burnout_Flag` are just re-labelled `Burnout_Score`,
so they are dropped as well (redundant).

In [ ]:
TARGET = "Startup_Failure_Flag"
LEAKY = ["Shutdown_Probability", "Shutdown_Risk", TARGET, "Burnout_Level", "Founder_Burnout_Flag"]

X = df.drop(columns=LEAKY)
y = df[TARGET]

cat_features = X.select_dtypes(include="object").columns.tolist()
num_features = X.select_dtypes(include="number").columns.tolist()
print("Categorical:", cat_features)
print("Numeric    :", len(num_features), "features")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_features),
])

models = {
    "Logistic Regression": Pipeline([("prep", preprocess),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "Hist Gradient Boosting": Pipeline([("prep", preprocess),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight="balanced"))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = {}
for name, m in models.items():
    s = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_scores[name] = s
    print(f"{name:25s} CV ROC-AUC = {s.mean():.4f} ± {s.std():.4f}")

### 7.1 Evaluate on the hold-out test set

In [ ]:
results = []
fitted = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    fitted[name] = m
    proba = m.predict_proba(X_test)[:, 1]
    pred = m.predict(X_test)
    results.append({"model": name, "ROC_AUC": roc_auc_score(y_test, proba),
                    "PR_AUC": average_precision_score(y_test, proba),
                    "accuracy": (pred == y_test).mean()})
pd.DataFrame(results).set_index("model")

In [ ]:
best_name = max(results, key=lambda r: r["ROC_AUC"])["model"]
best = fitted[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best.predict(X_test), target_names=["Survives", "Fails"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, display_labels=["Survives", "Fails"],
                                      cmap="Blues", ax=ax[0]); ax[0].set_title("Confusion matrix")
for name, m in fitted.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, name=name, ax=ax[1])
ax[1].plot([0, 1], [0, 1], "k--", lw=1); ax[1].set_title("ROC curves")
plt.tight_layout(); plt.show()

### 7.2 What drives startup failure? (permutation importance)

In [ ]:
perm = permutation_importance(best, X_test, y_test, scoring="roc_auc", n_repeats=5,
                              random_state=RANDOM_STATE, n_jobs=-1)
imp = (pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False).head(15))
plt.figure(figsize=(8, 5))
sns.barplot(x=imp.values, y=imp.index, color="#3E7CB1")
plt.title(f"Top 15 features by permutation importance ({best_name})"); plt.xlabel("drop in ROC-AUC")
plt.tight_layout(); plt.show()

## 8. Export final files

In [ ]:
final_path = OUTPUT_DIR / "startup_founder_burnout_2026_final.csv"
df.to_csv(final_path, index=False)

db_path = OUTPUT_DIR / "startup_founder_burnout_2026.db"
with sqlite3.connect(db_path) as con:
    df.to_sql("founders_final", con, index=False, if_exists="replace")

print("Saved:", final_path, "|", db_path)
print("Final shape:", df.shape)

## Key takeaways
* Data is clean: no missing values, no duplicates, valid ranges and consistent derived labels.
* `Shutdown_Risk` / `Startup_Failure_Flag` come straight from `Shutdown_Probability`. Never use them as model inputs.
* Burnout, employee turnover, co-founder conflict and long hours push risk up, while runway, product-market fit and work-life balance push it down.